<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/courses/c35-header.png" alt="Nextia Learning · Forecasting and Time Series: Predict What Comes Next" width="100%">

# New products with no history

**[New products with no history](https://learning.nextia-ai.com/courses/forecasting/m06/new-products-with-no-history/)** · Forecasting and Time Series: Predict What Comes Next · Module 6, lesson 3 of 4 · [learning.nextia-ai.com](https://learning.nextia-ai.com)

**You will learn to** forecast a product that has only six weeks of sales: from its own recent days, from its category's curve and from one global LightGBM model, and to see how much you can trust each.

| | |
|---|---|
| **Time** | About 35 minutes, of which about 20 seconds of waiting for LightGBM |
| **Needs** | An internet connection for the setup cells. No account and no key. The first cell also installs mlforecast and LightGBM. |
| **You should know** | WAPE and bias, from [Measure the error that matters](https://learning.nextia-ai.com/courses/forecasting/m03/measure-the-error-that-matters/). A global LightGBM model, from [Gradient boosting with LightGBM](https://learning.nextia-ai.com/courses/forecasting/m05/gradient-boosting-with-lightgbm/). |
| **Tested** | Python 3.12 with pandas 2.3.3, statsforecast 2.1.1, mlforecast 1.1.0, hierarchicalforecast 1.5.3, LightGBM 4.7.0 and matplotlib 3.11.2, in Jupyter on a laptop CPU (Apple M4, 16 GB), from a fresh start (*Restart and run all*) |

### How to use this notebook

This notebook has every query, task and check of the lesson. The [lesson page](https://learning.nextia-ai.com/courses/forecasting/m06/new-products-with-no-history/) has the full explanations, the diagrams and the knowledge checks that count toward your certificate. Keep both open.

1. Run the cells in order, from the top. Press **Shift+Enter** to run a cell and move to the next one.
2. At a **Predict** note, write your prediction down before you run the cell.
3. At a **Your turn** note, write your answer in the cell, run it, then run the check cell below it.
4. If you are stuck, open the [solution notebook](https://github.com/Nextia-Intelligence/nextia-learning-labs/blob/main/C35/M06-L03-new-products-with-no-history/new-products-with-no-history-solution.ipynb).

## Setup: install the packages

Run the next cell. It installs the versions that the course uses: pandas 2.3.3, statsforecast 2.1.1, mlforecast 1.1.0, lightgbm 4.7.0, scikit-learn 1.9.1, and matplotlib for the charts. It needs an internet connection and takes about a minute the first time.

You should see one line per package, with the versions above. If you see other versions, restart the kernel (in Colab: **Runtime › Restart session**; in Kaggle: **Run › Restart & clear cell outputs**), then run this cell again. Restarting is needed when an older version was already loaded.

In [ ]:
%pip install -q --disable-pip-version-check pandas==2.3.3 statsforecast==2.1.1 mlforecast==1.1.0 lightgbm==4.7.0 scikit-learn==1.9.1 matplotlib==3.11.2
import warnings; warnings.filterwarnings("ignore")
import pandas; print("pandas", pandas.__version__)
import statsforecast; print("statsforecast", statsforecast.__version__)
import mlforecast; print("mlforecast", mlforecast.__version__)
import lightgbm; print("lightgbm", lightgbm.__version__)
import sklearn; print("scikit-learn", sklearn.__version__)

## Setup: get the practice data

Run the next cell. It downloads the Larkfield data of this course (about 2 MB) into a folder `larkfield-forecasting/data`, and the course's backtest helper `backtest.py`, and checks each file's checksum. It needs an internet connection and no account, and takes about 10 seconds.

You should see `Ready: 403758 rows of daily sales (200 products, 3 regions), from 2024-04-01 to 2026-03-29.` If you run the cell again, it uses the files that are already there.

If the cell shows an error, read its last line:

- `Could not download`: check your internet connection, then run the cell again. In Kaggle, turn on **Internet** in the notebook settings.
- `wrong checksum`: delete the folder `larkfield-forecasting` (in Colab: the **Files** panel on the left), then run the cell again.

The Larkfield data is synthetic: we made it for this course. It has no real people, products or prices.

In [ ]:
# Setup: download the course data into larkfield-forecasting/data/ and check it.
import gzip, hashlib, os, shutil, subprocess, urllib.request
from pathlib import Path

LABS = "https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/C35/"
FILES = {
    "calendar.csv": "a5d1d5a3b640a983de72a30fca0e627a4ef23e797b1d80619e9ffcaadf118844",
    "demand.csv.gz": "8b39303985cbdb9bb1f3c705cd927ed6aaeb6f9665bd56ad4c3f8cb2b8578c99",
    "orders.csv": "bd8059fd73fe294daa157f8a6b6ae3f58319fa644eaa7ff0d31122b02cf05e30",
    "price_changes.csv": "889b258be4e933b87065d81d0771d6cd9b630ee90223b0a6ab074129c42153cb",
    "products.csv": "198b0a3de6eea8128af68f3cbc3a5c6acd3a64ac854fff18d37660857be8cfbf",
    "promotions.csv": "ca147959faa4c36c757742de7121976ffe79d59b53049dbbc8337a7547541f4a",
    "roster_hourly.csv": "6c2e0e96881b36ded52ed19458515c71f025288ddaebd589b50c82a8ad7fa69b",
    "tickets_daily.csv": "36f7067db060fdd2bd084c408c0622370a012773849f243020336f1fc678b31d",
    "tickets_hourly.csv.gz": "43f7fbb3e360a8e4a0cb4e101ea6a6242a76f0c37204d76ee9413ea3adbedd92",
    "weather.csv": "43a199a8b235940e291195bfd026901b62a10189b29f3a3ad9527b5382544276",
    "backtest.py": "764ecd38c6e96632be6f0e139e05eadd7b07d14a4ece73d881fc7559c2fa203a",
}
HERE = Path.cwd()
# The notebook works in its own folder, marked with a file, so it never
# writes into your own project if you open the notebook there.
if Path(".nextia-notebook").exists():  # the cell ran before: start from the parent folder
    os.chdir("..")
    HERE = Path.cwd()
PROJECT = Path("larkfield-forecasting").resolve()
(PROJECT / "data").mkdir(parents=True, exist_ok=True)
(PROJECT / ".nextia-notebook").touch()

def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

def download(url, path):
    try:
        urllib.request.urlretrieve(url, path)
    except OSError:
        # Python without certificates (python.org on macOS): try curl.
        if subprocess.run(["curl", "-fsSL", "-o", str(path), url]).returncode:
            raise SystemExit(f"Could not download {url}. Check your internet connection.")

def fetch(name, url, path, expected):
    """Use a copy next to the notebook if there is one, or download it."""
    if path.exists() and sha256(path) == expected:
        return
    if (HERE / name).exists():
        shutil.copy(HERE / name, path)
    else:
        download(url, path)
    if sha256(path) != expected:
        raise SystemExit(f"{name} has the wrong checksum. Delete the folder larkfield-forecasting and run again.")

for name, expected in FILES.items():
    if name == "backtest.py":
        fetch(name, LABS + name, PROJECT / name, expected)
    else:
        fetch(name, LABS + "data/" + name, PROJECT / "data" / name, expected)

os.chdir(PROJECT)  # work in the project folder, as in the lesson
with gzip.open("data/demand.csv.gz", "rt") as f:
    header = f.readline()
    rows = [line.split(",", 3)[:3] for line in f]
print(f"Ready: {len(rows)} rows of daily sales ({len({r[2] for r in rows})} products, {len({r[1] for r in rows})} regions), "
      f"from {min(r[0] for r in rows)} to {max(r[0] for r in rows)}.")

## Setup: helpers

The next cell loads pandas as `pd`, numpy as `np` and the course's backtest helper as `bt`, and defines the check helpers. They tell you if your answer is right, without showing the answer:

- `expect(what, yours, expected)` compares one of your numbers with the expected one, and says "too high" or "too low".
- `expect_hash(what, yours, expected)` compares any result with a hidden fingerprint of the expected one.

Run the cell. It prints nothing.

In [ ]:
# Helpers: the backtest helper, and checks that do not show the answer.
import hashlib, json, logging, os, warnings
os.environ["NIXTLA_ID_AS_COL"] = "1"
import numpy as np
import pandas as pd
import backtest as bt
pd.set_option("display.precision", 3)
pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 140)
warnings.filterwarnings("ignore")
logging.getLogger("lightning").setLevel(logging.ERROR)

def fingerprint(value):
    def norm(v):
        if hasattr(v, "to_dict") and hasattr(v, "columns"):
            return {"columns": [str(c) for c in v.columns], "rows": [norm(list(r)) for r in v.itertuples(index=False)]}
        if hasattr(v, "tolist"):
            return norm(v.tolist())
        if isinstance(v, dict):
            return {str(k): norm(x) for k, x in sorted(v.items(), key=lambda kv: str(kv[0]))}
        if isinstance(v, (list, tuple)):
            return [norm(x) for x in v]
        if isinstance(v, bool) or v is None or isinstance(v, int):
            return v
        if isinstance(v, float):
            return None if v != v else round(v, 4)
        if hasattr(v, "item"):
            return norm(v.item())
        return str(v)
    return hashlib.sha256(json.dumps(norm(value), sort_keys=True).encode()).hexdigest()[:16]

def expect(what, yours, expected, tolerance=1e-6):
    """Compare one number or value with the expected one."""
    if isinstance(yours, (np.integer, np.floating)):
        yours = yours.item()
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
        return
    if isinstance(expected, float) and isinstance(yours, (int, float)):
        same = abs(yours - expected) <= tolerance * max(1, abs(expected))
    else:
        same = yours == expected
    if same:
        print(f"Check passed: {what} is {yours}.")
    elif isinstance(expected, (int, float)) and isinstance(yours, (int, float)):
        print(f"Not yet: {what} is {yours}, which is too {'high' if yours > expected else 'low'}.")
    else:
        print(f"Not yet: {what} is {yours!r}, which is not the expected value.")

def expect_hash(what, yours, expected):
    """Compare any result with the fingerprint of the expected one."""
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
    elif fingerprint(yours) == expected:
        print(f"Check passed: {what} is right.")
    else:
        print(f"Not yet: {what} is not the expected result. Read the task again, and check each step.")

def load_sales(products=None):
    """Daily sales per product, the three regions added up, in the long format of the backtest:
    unique_id (the product), ds (the day), y (units sold), price, on_promotion, stock_out.
    The missing day 2025-10-19 gets the mean of the same weekday one week before and after."""
    d = pd.read_csv("data/demand.csv.gz", parse_dates=["date"])
    if products is not None:
        d = d[d["product_id"].isin(products)]
    g = d.groupby(["product_id", "date"], as_index=False).agg(
        y=("sales", "sum"), price=("price", "mean"), on_promotion=("on_promotion", "max"), stock_out=("stock_out", "max"))
    out = []
    for uid, s in g.rename(columns={"product_id": "unique_id", "date": "ds"}).groupby("unique_id"):
        s = s.set_index("ds").reindex(pd.date_range(s["ds"].min(), s["ds"].max(), freq="D"))
        gap = s["y"].isna()
        before, after = s["y"].shift(7), s["y"].shift(-7)
        s.loc[gap, "y"] = ((before + after) / 2).fillna(before).fillna(after)[gap].round()
        s = s.ffill()
        s["unique_id"] = uid
        out.append(s.rename_axis("ds").reset_index())
    return pd.concat(out, ignore_index=True)[["unique_id", "ds", "y", "price", "on_promotion", "stock_out"]]

# The course backtest (the same in every module): 13 origins, 28 days apart, the last one 2026-03-15.
ORIGINS = bt.origins("2026-03-15", n=13, every="28D")
BACKTEST_PRODUCTS = ['P001', 'P002', 'P005', 'P006', 'P009', 'P013', 'P018', 'P024', 'P025', 'P029', 'P032', 'P039', 'P045', 'P057', 'P068', 'P075', 'P077', 'P078', 'P080', 'P081', 'P087', 'P088', 'P093', 'P095', 'P108', 'P110', 'P121', 'P130', 'P133', 'P136', 'P137', 'P141', 'P145', 'P147', 'P148', 'P156', 'P158', 'P159', 'P166', 'P169', 'P172', 'P174', 'P176', 'P183', 'P184', 'P186', 'P191', 'P192', 'P194', 'P199']
PRICES = pd.read_csv("data/products.csv").set_index("product_id")["base_price"].to_dict()

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

## 1. The category curve by hand

A new lamp sold 30 units in its first 2 weeks. In the same 2 weeks, the average old lamp of its category sold 20. So the new lamp sells 30 ÷ 20 = 1.5 times as much as an average lamp. Last week, the average old lamp sold 2, 2, 3, 3, 4, 6 and 5 units (Monday to Sunday). The **category curve** forecast is that shape times 1.5.

> **Predict.** What does the category curve forecast for next Saturday? Then run the cell.

In [ ]:
new_lamp_sold, category_sold = 30, 20
ratio = new_lamp_sold / category_sold
category_last_week = np.array([2, 2, 3, 3, 4, 6, 5])
print("ratio:", ratio, "| next week:", (ratio * category_last_week).tolist())

> **Check.** You should see `ratio: 1.5 | next week: [3.0, 3.0, 4.5, 4.5, 6.0, 9.0, 7.5]`. Saturday: 9 units.

## 2. The new products of Larkfield

19 products started after the first day of the data and have at least six weeks and 14 more days of sales. We forecast each one on its **41st day after launch**: six weeks of history, as Kenji has when he plans the first full order.

In [ ]:
sales = load_sales()                         # national daily sales of all 200 products
products = pd.read_csv("data/products.csv", parse_dates=["launch_date"])
new = products[products["launch_date"] > "2024-04-01"].copy()
new["origin"] = new["launch_date"] + pd.Timedelta(days=41)
new = new[new["origin"] + pd.Timedelta(days=14) <= pd.Timestamp("2026-03-29")]
old_ids = set(products.loc[products["launch_date"] == "2024-04-01", "product_id"])
sales = sales.merge(products[["product_id", "category"]].rename(columns={"product_id": "unique_id"}), on="unique_id")
print(len(new), "new products:", new["product_id"].tolist())

> **Check.** You should see `19 new products:` and their IDs, from `P007` (a hammock) to `P180` (vacuum bags).

## 3. Three simple forecasts

For each new product, at its origin:

- **own mean**: the mean of its last 14 days, every day the same;
- **seasonal naive**: its last week, repeated;
- **category curve**: the average old product of the same category in the last week, times the ratio of section 1 (here over six weeks).

> **Predict.** The category curve uses all six weeks for the ratio, and a new product starts slowly. Will its forecasts be too high or too low on average? Then run the cell.

In [ ]:
def window(df, start, end):
    return df[(df["ds"] > start) & (df["ds"] <= end)]

results = {"own mean": [], "seasonal naive": [], "category curve": []}
for p in new.itertuples():
    mine = sales[sales["unique_id"] == p.product_id]
    hist = window(mine, p.origin - pd.Timedelta(days=42), p.origin)["y"].to_numpy()
    real = window(mine, p.origin, p.origin + pd.Timedelta(days=14))["y"].to_numpy()
    others = sales[(sales["category"] == p.category) & sales["unique_id"].isin(old_ids)]
    cat_6_weeks = window(others, p.origin - pd.Timedelta(days=42), p.origin).groupby("ds")["y"].mean()
    cat_last_week = window(others, p.origin - pd.Timedelta(days=7), p.origin).groupby("ds")["y"].mean().to_numpy()
    ratio = hist.sum() / cat_6_weeks.sum()
    results["own mean"].append((real, np.full(14, hist[-14:].mean())))
    results["seasonal naive"].append((real, np.tile(hist[-7:], 2)))
    results["category curve"].append((real, np.tile(cat_last_week, 2) * ratio))

def wape_bias(pairs):
    y = np.concatenate([a for a, _ in pairs]); f = np.concatenate([b for _, b in pairs])
    return round(float(100 * np.abs(y - f).sum() / y.sum()), 1), round(float(100 * (f - y).sum() / y.sum()), 1)

for name, pairs in results.items():
    print(f"{name:15} WAPE % and bias %:", wape_bias(pairs))

> **Check.** You should see `own mean` 35.7 and +2.9, `seasonal naive` 47.9 and +10.8, `category curve` 35.2 and −12.7.

The category curve has the lowest WAPE, but it is 12.7% too low. Its ratio counts the slow first weeks after the launch, so it underestimates a product that has now reached its normal level.

## 4. One global LightGBM model

A global model learns from all 200 products at once. Its inputs: the age of the product in days, the weekday, the category, the price, the promotion flag and the sales of 14, 21 and 28 days before. It learns again at each new product's origin, from every day up to it.

> **Predict.** Will the global model beat the category curve? Then run the cell (about 20 seconds: 19 models).

In [ ]:
import lightgbm as lgb

d = pd.read_csv("data/demand.csv.gz", parse_dates=["date"])
feat = d.groupby(["product_id", "date"], as_index=False).agg(y=("sales", "sum"), price=("price", "mean"),
                                                             promo=("on_promotion", "max"))
feat = feat.merge(products[["product_id", "category", "launch_date"]], on="product_id")
feat["age"] = (feat["date"] - feat["launch_date"]).dt.days
feat["dow"] = feat["date"].dt.dayofweek
feat["cat"] = feat["category"].astype("category").cat.codes
feat = feat.sort_values(["product_id", "date"])
for lag in (14, 21, 28):
    feat[f"lag{lag}"] = feat.groupby("product_id")["y"].shift(lag)
FEATURES = ["age", "dow", "cat", "price", "promo", "lag14", "lag21", "lag28"]

results["LightGBM global"] = []
for p in new.itertuples():
    train = feat[feat["date"] <= p.origin]
    model = lgb.LGBMRegressor(n_estimators=300, learning_rate=0.05, num_leaves=31, random_state=35, n_jobs=2,
                              verbose=-1).fit(train[FEATURES], train["y"])
    test = feat[(feat["product_id"] == p.product_id) & (feat["date"] > p.origin)
                & (feat["date"] <= p.origin + pd.Timedelta(days=14))]
    results["LightGBM global"].append((test["y"].to_numpy(), np.clip(model.predict(test[FEATURES]), 0, None)))
print("LightGBM global WAPE % and bias %:", wape_bias(results["LightGBM global"]))

> **Check.** You should see `LightGBM global WAPE % and bias %: (45.3, 6.6)`. (The demand table has no row for the missing day, 19 October 2025, so two products are scored on 13 days here.)

The global model is worse than both simple forecasts. It learned mostly from old products with two years of history. Their lags say a lot; a new product's lags say little, and some are missing.

## 5. How much can you trust six weeks?

In [ ]:
per_product = pd.DataFrame({name: [round(100 * np.abs(a - b).sum() / a.sum(), 1) for a, b in pairs]
                            for name, pairs in results.items()}, index=new["product_id"])
print(per_product.to_string())
print("best per product:", per_product.idxmin(axis=1).value_counts().to_dict())

> **Check.** You should see 19 rows. The best method per product: `own mean` 8, `category curve` 8, `seasonal naive` 2, `LightGBM global` 1. The hammock P007 has a WAPE of 18.3% with the category curve; the parasol base P027 is above 95% with every method.

No method is safe for every new product. The spread is wide: from about 20% to more than 100% for the same method.

## 6. Your turn: a ratio from the last two weeks

The category curve is too low because its ratio counts the slow start. Change one thing: compute the ratio over the **last 14 days** only (the new product's last 14 days ÷ the category's mean over the same 14 days).

> **Your turn.** Compute the forecasts of this new category curve for the 19 products, and put its WAPE % (rounded to 1 decimal, the first value of `wape_bias()`) in `wape_recent`. Then run the check cell.

In [ ]:
wape_recent = ...  # the WAPE % of the category curve with a 14-day ratio

In [ ]:
expect('wape_recent', wape_recent, 38.1, tolerance=0.001)

With a ratio from the last two weeks, the WAPE is 38.1% and the bias +9.4%. The bias moved from −12.7% to +9.4%: now the forecasts are too high. And the WAPE got worse: 38.1% against 35.2%. A shorter window removes the slow start, but two weeks of one new product are few and noisy. Check the WAPE and the bias together before you keep a change.

## Next

You can now forecast a product with six weeks of history and say how much to trust it. The [lesson page](https://learning.nextia-ai.com/courses/forecasting/m06/new-products-with-no-history/) has the diagrams and the knowledge checks that count toward your certificate.

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

**Next lesson:** [Breaks and changes](https://learning.nextia-ai.com/courses/forecasting/m06/breaks-and-changes/) · **This lesson:** [read it on the site](https://learning.nextia-ai.com/courses/forecasting/m06/new-products-with-no-history/)

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/notebook-footer-light.png" alt="Nextia Learning · learning.nextia-ai.com" width="100%">

Free, practical AI courses at [learning.nextia-ai.com](https://learning.nextia-ai.com). The practice data is synthetic and has no real people.